#  Predator 🦊 - 🐰 Prey Dynamics: The Lotka-Volterra Model
### Lesson 2, Section 1

In this notebook we will:
1. Introduce the Lotka-Volterra equations and their biological meaning
2. Implement two numerical solvers: **Euler method** and **SciPy's solve_ivp**
3. Visualise populations **over time**
4. Draw the **vector field** in phase space
5. Plot the **phase diagram** (trajectories for different initial conditions)
6. Explore equilibrium points analytically
7. 🧪 **Student tasks** — modify the model and explore!

---

## 0 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp

# nicer plots
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})
print('All imports OK ✓')

---
## 1 · The Mathematical Model

The **Lotka-Volterra** equations describe two interacting species:

$$\frac{dN}{dt} = rN - \beta NP$$

$$\frac{dP}{dt} = m\beta NP - dP$$

| Symbol | Meaning | Typical sign of effect |
|--------|---------|------------------------|
| $N$ | prey population (e.g. rabbits) | — |
| $P$ | predator population (e.g. foxes) | — |
| $r$ | prey intrinsic growth rate | $+$ |
| $\beta$ | attack efficiency of predator | $+/-$ |
| $m$ | conversion efficiency (prey → predator) | $+$ |
| $d$ | predator death rate | $-$ |

### Verbal interpretation
- **Prey** grow exponentially ($rN$) but are removed by encounters with predators ($-\beta NP$).
- **Predators** grow by eating prey ($+m\beta NP$) but die at rate $d$ when food is scarce.
- The product $NP$ captures random encounters in a well-mixed environment.

> ⚠️ **Assumption check**: the model assumes unlimited prey growth, no prey refuge, no predator saturation, and a homogeneous habitat. These are simplified but useful starting points.

---
## 2 · Model Parameters & Initial Conditions

Define everything in one place so that later sections automatically update when you change values here.

In [ ]:
# ── Model parameters ─────────────────────────────────────────────────────────
r    = 1.0   # prey growth rate
beta = 0.1   # attack efficiency
m    = 0.05  # conversion efficiency
d    = 0.5   # predator death rate

# ── Initial conditions ────────────────────────────────────────────────────────
N0 = 40.0    # initial prey
P0 = 9.0     # initial predators

# ── Simulation time ───────────────────────────────────────────────────────────
T  = 60.0    # total time

# ── Analytical equilibrium ────────────────────────────────────────────────────
N_star = d / (m * beta)   # coexistence prey equilibrium
P_star = r / beta         # coexistence predator equilibrium

print(f'Parameters:  r={r}, β={beta}, m={m}, d={d}')
print(f'Coexistence equilibrium:  N* = {N_star:.1f},  P* = {P_star:.1f}')

---
## 3 · Numerical Solvers

### 3a · Euler Method (manual implementation)

The Euler method replaces the continuous derivative with a finite-difference step:

$$N_{t+\Delta t} = N_t + \Delta t\,(rN_t - \beta N_t P_t)$$
$$P_{t+\Delta t} = P_t + \Delta t\,(m\beta N_t P_t - d P_t)$$

> 💡 **Intuition**: at each tick we "push" both populations forward by their current rate of change.

In [ ]:
def euler_lotka_volterra(r, beta, m, d, N0, P0, T, dt=0.01):
    """Solve Lotka-Volterra with the explicit Euler method."""
    steps = int(T / dt)
    t = np.linspace(0, T, steps)
    N = np.zeros(steps)
    P = np.zeros(steps)
    N[0], P[0] = N0, P0

    for i in range(1, steps):
        dN = r * N[i-1] - beta * N[i-1] * P[i-1]
        dP = m * beta * N[i-1] * P[i-1] - d * P[i-1]
        N[i] = N[i-1] + dt * dN
        P[i] = P[i-1] + dt * dP
        # guard against negative populations (numerical artefact)
        N[i] = max(N[i], 0.0)
        P[i] = max(P[i], 0.0)

    return t, N, P

t_euler, N_euler, P_euler = euler_lotka_volterra(r, beta, m, d, N0, P0, T, dt=0.01)
print(f'Euler: {len(t_euler)} time steps, dt=0.01')

### 3b · SciPy `solve_ivp` (Runge-Kutta RK45)

A modern adaptive-step-size solver. More accurate than Euler for the same computational cost.

In [ ]:
def lotka_volterra_ode(t, state, r, beta, m, d):
    """RHS of the Lotka-Volterra system for solve_ivp."""
    N, P = state
    dN_dt = r * N - beta * N * P
    dP_dt = m * beta * N * P - d * P
    return [dN_dt, dP_dt]

t_eval = np.linspace(0, T, 3000)   # points where we want the solution stored

sol = solve_ivp(
    fun=lotka_volterra_ode,
    t_span=(0, T),
    y0=[N0, P0],
    args=(r, beta, m, d),
    method='RK45',
    t_eval=t_eval,
    rtol=1e-8, atol=1e-10,
)

t_rk   = sol.t
N_rk   = sol.y[0]
P_rk   = sol.y[1]
print(f'RK45 solver status: {"success" if sol.success else sol.message}')

---
## 4 · Populations Over Time

Plot both solvers together to see how they compare — and to appreciate that the Euler method **drifts** slightly over long periods.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# ── Prey ─────────────────────────────────────────────────────────────────────
axes[0].plot(t_rk,    N_rk,    color='steelblue',  lw=2,   label='RK45 (SciPy)')
axes[0].plot(t_euler, N_euler, color='steelblue',  lw=1.2, ls='--', alpha=0.7, label='Euler')
axes[0].axhline(N_star, color='steelblue', ls=':', alpha=0.5, label=f'Equilibrium N*={N_star:.0f}')
axes[0].set_ylabel('Prey population $N$')
axes[0].set_title('Lotka-Volterra: Populations Over Time')
axes[0].legend(loc='upper right')
axes[0].set_ylim(bottom=0)

# ── Predators ─────────────────────────────────────────────────────────────────
axes[1].plot(t_rk,    P_rk,    color='firebrick', lw=2,   label='RK45 (SciPy)')
axes[1].plot(t_euler, P_euler, color='firebrick', lw=1.2, ls='--', alpha=0.7, label='Euler')
axes[1].axhline(P_star, color='firebrick', ls=':', alpha=0.5, label=f'Equilibrium P*={P_star:.0f}')
axes[1].set_ylabel('Predator population $P$')
axes[1].set_xlabel('Time')
axes[1].legend(loc='upper right')
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print('Observation: prey peaks BEFORE predator peaks — the predator lags behind the prey.')

---
## 5 · Vector Field in Phase Space

The **vector field** assigns an arrow $\vec{v}(N,P) = \left(\frac{dN}{dt},\, \frac{dP}{dt}\right)$ to every point in the $(N, P)$ plane.  
Arrow direction → where the system is heading.  
Arrow length → how fast the system changes.

The two equilibria are:
- $(0, 0)$ — **extinction** (trivial)
- $\left(\dfrac{d}{m\beta},\, \dfrac{r}{\beta}\right)$ — **coexistence**

In [ ]:
# ── Grid for the vector field ─────────────────────────────────────────────────
N_max, P_max = 220, 20
n_grid = 20
N_grid = np.linspace(1, N_max, n_grid)
P_grid = np.linspace(1, P_max, n_grid)
NN, PP = np.meshgrid(N_grid, P_grid)

dN = r * NN - beta * NN * PP
dP = m * beta * NN * PP - d * PP

# normalise arrow lengths for readability (keep direction only)
magnitude = np.sqrt(dN**2 + dP**2)
magnitude[magnitude == 0] = 1          # avoid division by zero
dN_norm = dN / magnitude
dP_norm = dP / magnitude

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# colour-map arrows by log-magnitude
quiv = ax.quiver(NN, PP, dN_norm, dP_norm,
                 color='black', pivot='mid', alpha=0.4,
                 scale=35, width=0.003)


# overlay one RK45 trajectory
ax.plot(N_rk, P_rk, color='blue', lw=3, label='Trajectory', alpha=0.85)
ax.plot(N0, P0, 'ko', ms=8, label='Initial condition')

# equilibrium points
ax.plot(N_star, P_star, 'g*', ms=14, label=f'Coexistence $({N_star:.0f},{P_star:.0f})$', zorder=5)

ax.set_xlabel('Prey $N$')
ax.set_ylabel('Predator $P$')
ax.set_title('Vector Field in Phase Space')
ax.legend(loc='upper right')
ax.set_xlim(0, N_max)
ax.set_ylim(0, P_max)
plt.tight_layout()
plt.show()

---
## 6 · Phase Diagram — Multiple Trajectories

A **phase diagram** shows *where* the system goes from different starting points, without showing *when*.  
Each closed loop corresponds to a different pair of initial conditions $(N_0, P_0)$.  
Notice all orbits circle the coexistence equilibrium — it is a **centre** (neutrally stable).

In [ ]:
# ── Several initial conditions ────────────────────────────────────────────────
N_max, P_max = 250, 20

initial_conditions = [
    (100, 5),
    (100, 6),
    (100, 7),
    (100, 8),
    (100, 9),
    (100, 10),
]

colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(initial_conditions)))

fig, ax = plt.subplots(figsize=(9, 7))

ax.quiver(NN, PP, dN_norm, dP_norm, alpha=0.3, color='grey', pivot='mid', scale=35, width=0.004)

for ic, col in zip(initial_conditions, colors):
    sol_ic = solve_ivp(
        lotka_volterra_ode,
        t_span=(0, T),
        y0=list(ic),
        args=(r, beta, m, d),
        method='RK45',
        t_eval=np.linspace(0, T, 3000),
        rtol=1e-8, atol=1e-10,
    )
    ax.plot(sol_ic.y[0], sol_ic.y[1], color=col, lw=1.8, alpha=0.9)
    ax.plot(*ic, 'o', color=col, ms=6)



# nullclines
N_range = np.linspace(0, N_max, 300)


ax.set_xlabel('Prey $N$', fontsize=13)
ax.set_ylabel('Predator $P$', fontsize=13)
ax.set_title('Phase Diagram — Closed Orbits Around Coexistence Equilibrium', fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(0, N_max)
ax.set_ylim(0, P_max)
plt.tight_layout()
plt.show()

print('Dashed lines: nullclines (where one population stops changing).')
print('Their intersection = coexistence equilibrium.')

---
## 7 · Equilibrium Analysis

We find equilibria by setting both derivatives to zero simultaneously.

**Prey nullcline** ($dN/dt = 0$):  
$rN - \beta NP = 0 \implies N=0$ or $P = r/\beta$

**Predator nullcline** ($dP/dt = 0$):  
$m\beta NP - dP = 0 \implies P=0$ or $N = d/(m\beta)$

The intersections give us the two fixed points.

In [ ]:
# Compute symbolically-style with numbers
print('=' * 50)
print('  EQUILIBRIUM ANALYSIS')
print('=' * 50)
print(f'\n1) Extinction point:   (N*, P*) = (0, 0)')
print(f'   Interpretation: trivial — both species absent.\n')

print(f'2) Coexistence point:  (N*, P*) = (d/(m·β),  r/β)')
print(f'   With our parameters:')
print(f'     N* = {d} / ({m} · {beta}) = {N_star:.2f}')
print(f'     P* = {r} / {beta}         = {P_star:.2f}')
print()
print('Biological insight:')
print(f'  • Higher predator death rate d → higher equilibrium prey N*')
print(f'  • Higher prey growth rate r   → higher equilibrium predator P*')
print(f'  • Prey equilibrium is INDEPENDENT of r; predator of d  — surprising?')

---
## 8 · Euler vs RK45: Stability Comparison

A key practical issue: the Euler method introduces numerical error that manifests as a **slowly spiraling-out trajectory** in phase space (artificial population growth).

In [ ]:
N_max, P_max = 350, 30

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, dt_val, title in zip(axes,[0.05, 0.005], ['Euler  dt = 0.05 (large step)', 'Euler  dt = 0.005 (small step)']):
    t_e, N_e, P_e = euler_lotka_volterra(r, beta, m, d, N0, P0, T, dt=dt_val)

    ax.plot(N_rk, P_rk, 'k-', lw=2, label='RK45 (reference)', alpha=0.9)
    ax.plot(N_e,  P_e,  'tomato', lw=1.5, ls='--', label=f'Euler Δt={dt_val}')
    ax.plot(N_star, P_star, 'g*', ms=12, zorder=5)
    ax.set_xlabel('Prey $N$'); ax.set_ylabel('Predator $P$')
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, N_max); ax.set_ylim(0, P_max)

plt.suptitle('Euler Method: Effect of Step Size on Trajectory Accuracy', fontweight='bold')
plt.tight_layout()
plt.show()

print('Left:  large dt → spiral outward (numerical drift).')
print('Right: small dt → closely matches the accurate RK45 solution.')

---
## 9 · Playground Dashboard

A single figure combining time-series + phase diagram + vector field for a concise overview.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp

# ── Parameters ────────────────────────────────────────────────────────────────
r    = 1.5   # prey growth rate
beta = 0.1   # attack efficiency
m    = 0.05  # conversion efficiency
d    = 0.5   # predator death rate

# ── Initial conditions ────────────────────────────────────────────────────────
N0 = 40.0    # initial prey
P0 = 9.0     # initial predators
T  = 60.0    # total time for simulation

# ── Model Definition ──────────────────────────────────────────────────────────
def lotka_volterra_ode(t, y, r, beta, m, d):
    """Defines the Lotka-Volterra system of differential equations."""
    N, P = y
    dN_dt = r * N - beta * N * P
    dP_dt = m * beta * N * P - d * P
    return [dN_dt, dP_dt]

# Calculate equilibrium points
N_star = d / (m * beta)
P_star = r / beta

# Solve for the main trajectory
sol_main = solve_ivp(lotka_volterra_ode, (0, T), [N0, P0], 
                     args=(r, beta, m, d), method='RK45', 
                     t_eval=np.linspace(0, T, 3000), rtol=1e-8)

t_rk = sol_main.t
N_rk, P_rk = sol_main.y

# Calculate limits for the phase diagram
N_max = max(140, np.max(N_rk) * 1.1)
P_max = max(30, np.max(P_rk) * 1.1)

# ── Plotting Dashboard ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1.3], wspace=0.25, hspace=0.3)

# ── (a) Prey over time ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t_rk, N_rk, color='steelblue', lw=2)
ax1.plot(0, N0, 'ko', ms=12, color='steelblue') # Mark the initial condition
ax1.axhline(N_star, ls=':', color='steelblue', alpha=0.5)
ax1.set_title('(a) Prey over time')
ax1.set_xlabel('Time')
ax1.set_ylabel('$N$ (prey)')
ax1.set_ylim(bottom=0)
ax1.legend()

# ── (b) Predator over time ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(t_rk, P_rk, color='firebrick', lw=2)
ax2.plot(0, P0, 'ko', ms=12, color='firebrick') # Mar
ax2.axhline(P_star, ls=':', color='firebrick', alpha=0.5)
ax2.set_title('(b) Predators over time')
ax2.set_xlabel('Time')
ax2.set_ylabel('$P$ (predators)')
ax2.set_ylim(bottom=0)
ax2.legend()

# ── (c) Phase diagram ─────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[:, 1])

# Plot the single trajectory that corresponds directly to the left plots
ax3.plot(N_rk, P_rk, color='black', lw=2)
ax3.plot(N0, P0, 'ko', ms=12, label='Start') # Mark the initial condition

# Mark Equilibrium lines
ax3.axvline(N_star, color='steelblue', ls='--', lw=1.2, alpha=0.7)
ax3.axhline(P_star, color='firebrick', ls='--', lw=1.2, alpha=0.7)

ax3.set_title('(c) Phase diagram')
ax3.set_xlabel('Prey $N$')
ax3.set_ylabel('Predator $P$')
ax3.legend(loc='upper right')
ax3.set_xlim(0, N_max)
ax3.set_ylim(0, P_max)

plt.suptitle('Lotka-Volterra Predator-Prey Dynamics', fontsize=16, fontweight='bold')
plt.show()